# 02 — Analysis: LST, NDVI, and Urban Heat Island Change

**Project:** Urban Heat Island Satellite Analysis — London, UK  
**Author:** Zari Syed  
**Date:** 2025

---

## Overview

This notebook performs the core analysis on the preprocessed Landsat data. It covers:

1. NDVI calculation for 2015 and 2023
2. Land Surface Temperature (LST) derivation with emissivity correction
3. Delta rasters (2023 − 2015) for both LST and NDVI
4. Borough-level zonal statistics
5. Identification of the top 5 hotspot boroughs by ΔLST
6. Pearson correlation between LST and NDVI

**Inputs:** `data/processed/*_london.tif`  
**Outputs:** Derived rasters in `data/processed/`, summary tables in `outputs/tables/`

## 1. Imports and Paths

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask as rio_mask
import geopandas as gpd
from shapely.geometry import mapping
from scipy import stats

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
PROCESSED    = PROJECT_ROOT / 'data'    / 'processed'
SHAPEFILES   = PROJECT_ROOT / 'shapefiles'
TABLES_OUT   = PROJECT_ROOT / 'outputs' / 'tables'
TABLES_OUT.mkdir(parents=True, exist_ok=True)

# ── Physical constants for LST calculation ─────────────────────────────────
# Collection 2 Level-2 ST scale factor and offset
ST_SCALE  = 0.00341802
ST_OFFSET = 149.0

# Emissivity parameters (Sobrino et al., 2004)
EPS_V = 0.985   # Vegetation emissivity
EPS_S = 0.960   # Soil/urban emissivity
C_CAV = 0.005   # Cavity effect term

# Planck function constant for TIRS Band 10
LAMBDA = 10.895e-6   # Effective wavelength (m)
RHO    = 1.438e-2    # h*c/sigma (m·K)

# SR scale factor and offset (Collection 2 Level-2)
SR_SCALE  = 0.0000275
SR_OFFSET = -0.2

NODATA = -9999.0

print("Setup complete.")

## 2. NDVI Calculation

NDVI is computed from surface reflectance Bands 4 (Red) and 5 (NIR):

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

Surface reflectance integer values are rescaled to physical reflectance (0–1) before the calculation.

In [ ]:
def load_band(path: Path, nodata: float = NODATA) -> tuple[np.ndarray, dict]:
    """Load a single-band raster, returning the array and profile."""
    with rasterio.open(path) as src:
        arr  = src.read(1).astype(np.float32)
        prof = src.profile.copy()
        src_nodata = src.nodata

    # Mask original nodata values
    if src_nodata is not None:
        arr = np.where(arr == src_nodata, np.nan, arr)
    else:
        arr = np.where(arr == nodata, np.nan, arr)

    prof.update({'dtype': 'float32', 'nodata': np.nan})
    return arr, prof


def calculate_ndvi(
    red_path : Path,
    nir_path : Path,
    year     : str,
    out_dir  : Path,
) -> tuple[np.ndarray, dict]:
    """
    Calculate NDVI from Collection 2 Level-2 surface reflectance bands.
    Rescales integer DN to reflectance, computes NDVI, saves GeoTIFF.
    Returns (ndvi_array, rasterio_profile).
    """
    red_raw, prof = load_band(red_path)
    nir_raw, _    = load_band(nir_path)

    # Rescale to surface reflectance
    red = red_raw * SR_SCALE + SR_OFFSET
    nir = nir_raw * SR_SCALE + SR_OFFSET

    # Clip to valid reflectance range [0, 1]
    red = np.clip(red, 0, 1)
    nir = np.clip(nir, 0, 1)

    # NDVI — suppress division by zero
    with np.errstate(invalid='ignore', divide='ignore'):
        ndvi = np.where(
            (nir + red) == 0,
            np.nan,
            (nir - red) / (nir + red)
        )

    # Save
    out_path = out_dir / f'ndvi_{year}_london.tif'
    with rasterio.open(out_path, 'w', **prof) as dst:
        dst.write(ndvi.astype(np.float32), 1)

    valid = ~np.isnan(ndvi)
    print(f"  NDVI {year} — min: {np.nanmin(ndvi):.3f}  max: {np.nanmax(ndvi):.3f}  "
          f"mean: {np.nanmean(ndvi):.3f}  valid px: {valid.sum():,}")
    return ndvi, prof


print("Calculating NDVI...")
ndvi_2015, prof_ref = calculate_ndvi(
    PROCESSED / '2015_red_bng_london.tif',
    PROCESSED / '2015_nir_bng_london.tif',
    '2015', PROCESSED
)
ndvi_2023, _ = calculate_ndvi(
    PROCESSED / '2023_red_bng_london.tif',
    PROCESSED / '2023_nir_bng_london.tif',
    '2023', PROCESSED
)

## 3. Land Surface Temperature (LST) Derivation

LST is derived in three steps:

**Step 1 — Brightness temperature:**  
$T_B = \text{ST\_B10} \times 0.00341802 + 149.0 \quad [K]$

**Step 2 — NDVI-based emissivity:**  
$F_v = \left(\frac{\text{NDVI} - \text{NDVI}_{\min}}{\text{NDVI}_{\max} - \text{NDVI}_{\min}}\right)^2$  
$\varepsilon = \varepsilon_v F_v + \varepsilon_s (1 - F_v) + C$

**Step 3 — LST inversion (Jiménez-Muñoz & Sobrino, 2003):**  
$\text{LST} = \frac{T_B}{1 + \left(\frac{\lambda T_B}{\rho}\right) \ln \varepsilon} - 273.15 \quad [°C]$

In [ ]:
def derive_lst(
    thermal_path : Path,
    ndvi         : np.ndarray,
    year         : str,
    out_dir      : Path,
    profile      : dict,
) -> np.ndarray:
    """
    Derive Land Surface Temperature (°C) from Landsat Collection 2
    ST_B10 and NDVI-based emissivity.
    Returns LST array in degrees Celsius.
    """
    # ── Step 1: Brightness temperature ────────────────────────────────────
    st_raw, _ = load_band(thermal_path)
    T_B = st_raw * ST_SCALE + ST_OFFSET   # Kelvin

    # ── Step 2: NDVI-based fractional vegetation cover and emissivity ──────
    ndvi_min = np.nanpercentile(ndvi, 2)
    ndvi_max = np.nanpercentile(ndvi, 98)

    Fv = np.clip(
        ((ndvi - ndvi_min) / (ndvi_max - ndvi_min)) ** 2,
        0.0, 1.0
    )
    emissivity = EPS_V * Fv + EPS_S * (1.0 - Fv) + C_CAV

    print(f"  {year} emissivity — min: {np.nanmin(emissivity):.4f}  "
          f"max: {np.nanmax(emissivity):.4f}  mean: {np.nanmean(emissivity):.4f}")

    # ── Step 3: LST inversion ──────────────────────────────────────────────
    with np.errstate(invalid='ignore', divide='ignore'):
        lst_K = T_B / (1.0 + (LAMBDA * T_B / RHO) * np.log(emissivity))

    lst_C = lst_K - 273.15   # Convert to Celsius

    # Propagate NaN from NDVI / thermal masks
    lst_C = np.where(np.isnan(ndvi) | np.isnan(T_B), np.nan, lst_C)

    # ── Save ──────────────────────────────────────────────────────────────
    out_path = out_dir / f'lst_{year}_london.tif'
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(lst_C.astype(np.float32), 1)

    print(f"  LST {year} (°C) — min: {np.nanmin(lst_C):.1f}  max: {np.nanmax(lst_C):.1f}  "
          f"mean: {np.nanmean(lst_C):.1f}")
    return lst_C


print("Deriving LST...")
lst_2015 = derive_lst(
    PROCESSED / '2015_thermal_bng_london.tif',
    ndvi_2015, '2015', PROCESSED, prof_ref
)
lst_2023 = derive_lst(
    PROCESSED / '2023_thermal_bng_london.tif',
    ndvi_2023, '2023', PROCESSED, prof_ref
)

## 4. Delta Rasters (2023 − 2015)

Pixel-wise differences reveal where LST has increased or decreased and how vegetation cover has changed over the study period.

In [ ]:
# ── ΔLST ──────────────────────────────────────────────────────────────────
delta_lst = lst_2023 - lst_2015

delta_lst_path = PROCESSED / 'delta_lst_2023_2015.tif'
with rasterio.open(delta_lst_path, 'w', **prof_ref) as dst:
    dst.write(delta_lst.astype(np.float32), 1)

print("ΔLST (2023 − 2015):")
print(f"  Mean ΔLST : {np.nanmean(delta_lst):+.2f} °C")
print(f"  Max  ΔLST : {np.nanmax(delta_lst):+.2f} °C")
print(f"  Min  ΔLST : {np.nanmin(delta_lst):+.2f} °C")
print(f"  Warmed px  : {(delta_lst > 0).sum():,}  ({(delta_lst > 0).mean()*100:.1f}%)")
print(f"  Cooled px  : {(delta_lst < 0).sum():,}  ({(delta_lst < 0).mean()*100:.1f}%)")

print()

# ── ΔNDVI ─────────────────────────────────────────────────────────────────
delta_ndvi = ndvi_2023 - ndvi_2015

delta_ndvi_path = PROCESSED / 'delta_ndvi_2023_2015.tif'
with rasterio.open(delta_ndvi_path, 'w', **prof_ref) as dst:
    dst.write(delta_ndvi.astype(np.float32), 1)

print("ΔNDVI (2023 − 2015):")
print(f"  Mean ΔNDVI : {np.nanmean(delta_ndvi):+.4f}")
print(f"  Max  ΔNDVI : {np.nanmax(delta_ndvi):+.4f}")
print(f"  Min  ΔNDVI : {np.nanmin(delta_ndvi):+.4f}")

## 5. Borough-Level Zonal Statistics

We aggregate mean LST, NDVI, and their deltas to each of the 33 London boroughs to identify spatial patterns in UHI change.

In [ ]:
# Load borough boundaries
boroughs_path = SHAPEFILES / 'london_boroughs.shp'
boroughs = gpd.read_file(boroughs_path).to_crs(epsg=27700)

print(f"Loaded {len(boroughs)} boroughs")
print(f"CRS: {boroughs.crs}")
boroughs[['NAME', 'geometry']].head()

In [ ]:
def zonal_mean(
    raster_path : Path,
    gdf         : gpd.GeoDataFrame,
    col_name    : str,
) -> gpd.GeoDataFrame:
    """
    Compute mean raster value within each polygon in gdf.
    Returns gdf with a new column `col_name`.
    """
    means = []
    with rasterio.open(raster_path) as src:
        for _, row in gdf.iterrows():
            geom = [mapping(row.geometry)]
            try:
                masked, _ = rio_mask(src, geom, crop=True, nodata=np.nan, filled=True)
                arr = masked[0].astype(np.float32)
                arr[arr == np.nan] = np.nan
                mean_val = np.nanmean(arr)
            except Exception:
                mean_val = np.nan
            means.append(mean_val)

    result = gdf.copy()
    result[col_name] = means
    return result


print("Computing zonal statistics (this may take a minute)...")

boroughs = zonal_mean(PROCESSED / 'lst_2015_london.tif',    boroughs, 'lst_2015')
boroughs = zonal_mean(PROCESSED / 'lst_2023_london.tif',    boroughs, 'lst_2023')
boroughs = zonal_mean(PROCESSED / 'ndvi_2015_london.tif',   boroughs, 'ndvi_2015')
boroughs = zonal_mean(PROCESSED / 'ndvi_2023_london.tif',   boroughs, 'ndvi_2023')
boroughs = zonal_mean(PROCESSED / 'delta_lst_2023_2015.tif', boroughs, 'delta_lst')
boroughs = zonal_mean(PROCESSED / 'delta_ndvi_2023_2015.tif', boroughs, 'delta_ndvi')

print("Done.")

## 6. Top 5 Hotspot Boroughs

Hotspot boroughs are defined as those with the greatest mean positive ΔLST — i.e., where surface temperatures have warmed most between 2015 and 2023.

In [ ]:
# Sort by ΔLST descending
cols = ['NAME', 'lst_2015', 'lst_2023', 'delta_lst', 'ndvi_2015', 'ndvi_2023', 'delta_ndvi']
summary = (
    boroughs[cols]
    .sort_values('delta_lst', ascending=False)
    .reset_index(drop=True)
)

print("Top 5 hotspot boroughs (highest ΔLST 2023–2015):")
print(summary.head(5).to_string(index=False, float_format='{:.2f}'.format))

print("\nBottom 5 boroughs (cooled or least warming):")
print(summary.tail(5).to_string(index=False, float_format='{:.2f}'.format))

In [ ]:
# Save full borough summary to CSV
csv_path = TABLES_OUT / 'borough_lst_ndvi_summary.csv'
summary.to_csv(csv_path, index=False, float_format='%.4f')
print(f"Saved borough summary: {csv_path}")

## 7. LST–NDVI Correlation Analysis

We compute Pearson's r between pixel-level LST and NDVI for each year, and between ΔLST and ΔNDVI. A strong negative correlation (r ≈ −0.7 to −0.9) is expected, reflecting the cooling role of vegetation.

In [ ]:
def pixel_correlation(
    arr_x : np.ndarray,
    arr_y : np.ndarray,
    label : str,
) -> dict:
    """
    Compute Pearson r between two raster arrays, excluding NaN pixels.
    Returns a dict with r, p-value, and n.
    """
    # Only valid pixels in both arrays
    valid = ~np.isnan(arr_x) & ~np.isnan(arr_y)
    x = arr_x[valid].ravel()
    y = arr_y[valid].ravel()

    r, p = stats.pearsonr(x, y)
    n = len(x)

    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f"  {label:<35}  r = {r:+.4f}   p = {p:.2e}  {sig}   n = {n:,}")

    return {'comparison': label, 'r': r, 'p_value': p, 'n': n}


print("Pearson correlation results:")
print("-" * 75)

results = [
    pixel_correlation(lst_2015,   ndvi_2015,  'LST 2015 vs NDVI 2015'),
    pixel_correlation(lst_2023,   ndvi_2023,  'LST 2023 vs NDVI 2023'),
    pixel_correlation(delta_lst,  delta_ndvi, 'ΔLST vs ΔNDVI (change)'),
]

# Save correlation table
corr_df = pd.DataFrame(results)
corr_path = TABLES_OUT / 'correlation_results.csv'
corr_df.to_csv(corr_path, index=False, float_format='%.6f')
print(f"\nSaved correlation results: {corr_path}")

---

## Summary

This notebook has produced:

| Output | Location |
|---|---|
| `lst_2015_london.tif` | `data/processed/` |
| `lst_2023_london.tif` | `data/processed/` |
| `ndvi_2015_london.tif` | `data/processed/` |
| `ndvi_2023_london.tif` | `data/processed/` |
| `delta_lst_2023_2015.tif` | `data/processed/` |
| `delta_ndvi_2023_2015.tif` | `data/processed/` |
| `borough_lst_ndvi_summary.csv` | `outputs/tables/` |
| `correlation_results.csv` | `outputs/tables/` |

Proceed to `03_visualisation.ipynb` to generate all figures.